# 21 – MCP Server (Tool Catalogue & Direct Invocation)

The MCP server exposes 10 tools via the Model Context Protocol (stdio or SSE transport).
This notebook calls the tool handlers directly — no MCP transport setup needed.

| Tool | Description |
|------|-------------|
| `governance_query` | Full multi-agent pipeline |
| `query_metrics` | Information agent |
| `search_knowledge_base` | Knowledge RAG agent |
| `get_metadata` | Metadata/Collibra agent |
| `manage_incidents` | Capacity/Jira agent |
| `manage_rules` | Rule agent |
| `get_system_status` | System health |
| `invalidate_cache` | Cache bust |
| `approve_action` | HITL approval |
| `ingest_document_url` | Document ingest |

In [ ]:
import sys, os, asyncio, json
sys.path.insert(0, os.path.abspath('../src'))
os.environ['ENABLE_MOCK'] = 'true'

In [ ]:
from mcp_server.server import call_tool, TOOLS

async def invoke(name: str, **kwargs):
    result = await call_tool(name, kwargs)
    text = result[0].text
    return json.loads(text)

print(f'Tools registered: {len(TOOLS)}')
for t in TOOLS:
    print(f"  {t.name:<30} {t.description[:60]}...")

## 1. governance_query — full pipeline

In [ ]:
result = asyncio.run(invoke(
    'governance_query',
    query='What is the DQ score for retention?',
    data_products=['retention'],
))

print('intent     :', result.get('intent'))
print('confidence :', result.get('confidence'))
print('summary    :', str(result.get('summary', ''))[:150])
print('anomalies  :', result.get('anomalies'))

## 2. query_metrics — information agent direct

In [ ]:
result = asyncio.run(invoke(
    'query_metrics',
    query='retention metrics',
    data_products=['retention'],
    time_range='last_30_days',
))

print('success  :', result.get('success'))
print('sources  :', result.get('sources'))
metrics = (result.get('data') or {}).get('metrics', {})
print('products :', list(metrics.keys()))

## 3. search_knowledge_base — RAG agent

In [ ]:
result = asyncio.run(invoke(
    'search_knowledge_base',
    query='What is the GRR threshold governance policy?',
))

print('success  :', result.get('success'))
knowledge = (result.get('data') or {}).get('knowledge', [])
print(f'docs found: {len(knowledge)}')
for doc in knowledge[:2]:
    print(f"  [{doc['topic']}]")

## 4. get_metadata — Collibra agent

In [ ]:
result = asyncio.run(invoke(
    'get_metadata',
    query='Who owns the bookings dataset?',
    data_products=['bookings'],
))

print('success:', result.get('success'))
data = result.get('data') or {}
if 'bookings' in data:
    print('owner  :', data['bookings'].get('owner'))
    print('DQ     :', data['bookings'].get('data_quality', {}).get('score'))

## 5. manage_incidents — Jira agent

In [ ]:
result = asyncio.run(invoke(
    'manage_incidents',
    query='Show open tickets for retention',
    data_products=['retention'],
))

print('success  :', result.get('success'))
tickets = (result.get('data') or {}).get('tickets', [])
print(f'tickets  : {len(tickets)}')
for t in tickets[:3]:
    print(f"  [{t.get('id')}] {t.get('summary', '')[:50]}")

## 6. manage_rules — rule agent

In [ ]:
result = asyncio.run(invoke(
    'manage_rules',
    query='list all rules',
))

print('success:', result.get('success'))
rules = result.get('data') or []
print(f'rules  : {len(rules)}')
for r in rules[:4]:
    print(f"  [{r['id']}] [{r['type']}] {r['name']}")

## 7. get_system_status

In [ ]:
result = asyncio.run(invoke('get_system_status'))

print('status    :', result.get('status'))
print('agents    :', [a['name'] for a in result.get('agents', [])])
print('redis_ok  :', result.get('redis_ok'))
print('mock_mode :', result.get('mock_mode'))
print('version   :', result.get('version'))

## 8. invalidate_cache

In [ ]:
result = asyncio.run(invoke('invalidate_cache', agent='information_agent'))
print('Result:', result)

result_all = asyncio.run(invoke('invalidate_cache', agent='all'))
print('All agents result:', result_all)

## 9. approve_action — HITL via MCP

In [ ]:
result = asyncio.run(invoke(
    'approve_action',
    thread_id='mcp-thread-123',
    query='show retention metrics',
))

print('intent  :', result.get('intent'))
print('summary :', str(result.get('summary', ''))[:100])

## 10. Tool schema inspection

In [ ]:
import json

for tool in TOOLS:
    required = tool.inputSchema.get('required', [])
    props = list(tool.inputSchema.get('properties', {}).keys())
    print(f'{tool.name:<30} required={required}  params={props}')